# 04 · Clasificación y priorización de leads

## Qué vamos a aprender y entregar

Este notebook responde al punto 4 de la prueba: asignar una prioridad explicable. Entrenamos **regresión logística y árboles pequeños**, los comparamos con reglas y orden de llegada, y documentamos incluso un resultado negativo. Tener un modelo entrenado no demuestra que sea útil.

**Recorrido:** fuentes → objetivo → variables y riesgos → separación temporal → entrenamiento → validación → prueba final → explicación → prioridad de los leads actuales → exportación y controles.

Los notebooks anteriores son `01_02_importacion_datos.ipynb` y `03_extraccion_ia_piloto.ipynb`. Sus resultados se leen sin modificarlos. No se llama a Gemini ni se usa la API key en este paso.

**Ejecución:** seleccionar el entorno `.venv`, instalar `requirements-notebook04.txt` y ejecutar todas las celdas en orden. El notebook encuentra la raíz desde la carpeta del proyecto o `notebooks`. Los artefactos propios se regeneran, con semilla y fechas fijas.

### Alcance de las conclusiones

Los datos son sintéticos. El histórico es una foto final, sin fecha del desenlace ni fecha de captura de las declaraciones. Una separación por fecha de registro ayuda, pero **no permite reconstruir exactamente qué información y qué etiquetas estaban disponibles en cada momento**. El experimento comercial es retrospectivo: no demuestra capacidad predictiva prospectiva. El modelo queda experimental hasta contar con esas fechas y nuevas observaciones.

> **Actualización:** la sección 14 fue ejecutada por el usuario y sus resultados fueron revisados. La conclusión está en 14.8. Las salidas de las secciones 1–13 corresponden al experimento inicial.

In [ ]:
from pathlib import Path
import json, hashlib, math, platform
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, precision_recall_curve

In [ ]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data/processed/historico_cierres.csv').exists())
DATA = ROOT / 'data/processed'
OUT = ROOT / 'outputs/clasificacion_04'
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
FRACCION_COLA = 0.20  # Escenario de capacidad; no es una capacidad suministrada por la empresa.
VERSION = '04-v1'
ENTRADAS = ['historico_cierres.csv', 'leads.csv', 'consultas_leads.csv',
            'extracciones_conversaciones_ia.json', 'conversaciones_utilizables_ia.json']
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
hashes_entrada = {p: sha(DATA / p) for p in ENTRADAS}
hist = pd.read_csv(DATA / 'historico_cierres.csv')
leads = pd.read_csv(DATA / 'leads.csv')
consultas = pd.read_csv(DATA / 'consultas_leads.csv')
extracciones = json.loads((DATA / 'extracciones_conversaciones_ia.json').read_text(encoding='utf-8'))
conversaciones = json.loads((DATA / 'conversaciones_utilizables_ia.json').read_text(encoding='utf-8'))
incidencias = []
pd.set_option('display.max_colwidth', 100)
display(pd.DataFrame({'fuente': ENTRADAS, 'filas': [len(hist), len(leads), len(consultas), len(extracciones), len(conversaciones)]}))

## 1. Qué significa la etiqueta que aprenderá el modelo

Una fila histórica representa un lead, no un mensaje. Definimos `y=1` cuando el resultado es **Cerrado**, y `y=0` cuando es **Perdido**. Los **Sin gestión no son negativos**: no conocemos su resultado tras atenderlos. Se apartan únicamente del entrenamiento y permanecen en la fuente y en el JSON de incidencias con el motivo.

Esto introduce una limitación: aprendemos sobre personas gestionadas, no sobre todas las personas que llegaron. No tenemos fecha de cierre para definir «compra en 30 días» ni para verificar maduración de etiquetas. Tampoco tenemos identificadores personales en el histórico para detectar si dos IDs pertenecen a una persona; comprobamos unicidad de la clave disponible.

Una predicción siempre se condiciona al proceso comercial de estos datos. No es una propiedad permanente del cliente ni una estimación del efecto causal de llamarlo.

In [ ]:
assert not hist.duplicated(['empresa_id', 'lead_id']).any(), 'Clave histórica repetida: revisar antes de entrenar.'
assert not leads.duplicated(['empresa_id', 'lead_consolidado_id']).any()
assert set(hist.desenlace.dropna()) <= {'Cerrado', 'Perdido', 'Sin gestión'}
hist['fecha'] = pd.to_datetime(hist.fecha_registro, format='%Y-%m-%d', errors='coerce')
sin_etiqueta = ~hist.desenlace.isin(['Cerrado', 'Perdido'])
sin_fecha = hist.fecha.isna()
for _, r in hist[sin_etiqueta | sin_fecha].iterrows():
    incidencias.append({'tipo': 'fuera_del_entrenamiento', 'empresa_id': r.empresa_id, 'lead_id': r.lead_id,
                        'motivo': 'Resultado no observado: no se convierte en perdido.' if sin_etiqueta.loc[r.name]
                                  else 'Fecha de registro inválida: no permite partición temporal.',
                        'accion': 'Conservar en la fuente; excluir solo de este experimento.'})
historico = hist[~sin_etiqueta & ~sin_fecha].copy()
historico['y'] = historico.desenlace.eq('Cerrado').astype(int)
display(hist.desenlace.value_counts().rename_axis('resultado').to_frame('registros'))
display(pd.crosstab(hist.fecha.dt.strftime('%Y-%m'), hist.desenlace))
display(pd.crosstab(hist.desenlace, hist.horas_al_primer_contacto.isna()).rename(columns={False: 'con_contacto', True: 'sin_contacto'}))
print(f'Tasa de cierre entre resultados observados: {historico.y.mean():.2%}')

## 2. Variables: decidir qué entra antes de mirar el resultado de prueba

| Variable | Decisión | Por qué |
|---|---|---|
| Canal | Experimento conservador y comercial | Describe la procedencia; también existe en consultas actuales. |
| Modelo de moto / SKU | Solo comercial | El histórico informa modelo **cotizado**, no necesariamente el modelo inicial. Es una aproximación al interés, con riesgo temporal y de cambio de significado. |
| Cuota inicial manifestada | Solo comercial | SI, NO y NO_INFORMA son distintos. No contamos importes porque el histórico no los tiene. |
| Forma de pago declarada | Solo comercial | Conservamos crédito, contado y no informado; no suponemos capacidad económica. |
| Solicitud de cita observada | Solo comercial | 1 cuando existe solicitud afirmativa, 0 cuando no está registrada. Cero no significa rechazo explícito. |
| Número de contactos | Excluir | Puede acumular gestiones posteriores; favorece a quienes ya recibieron atención. |
| Horas al primer contacto | Excluir | No existe al ingreso; fechas actuales sin hora impiden una equivalencia fiable. Es útil para analizar servicio, no para este modelo. |
| Estado actual / desenlace | Excluir de X | El desenlace es la respuesta; el estado puede revelar avances posteriores. |
| Precio | Excluir | Es redundante con el SKU en este catálogo y no es presupuesto del cliente. |
| Empresa, punto de venta, IDs, datos personales | Excluir de X | La empresa se usa para vincular, auditar y separar colas; no para puntuar a sus clientes. IDs no describen intención. |
| Intención, objeción, presupuesto, cotización | Contexto visible, fuera del modelo | No existen variables históricas equivalentes para aprender sus pesos. No inventamos etiquetas. |

**Dos preguntas diferentes:** ¿hay señal solo en el canal de entrada? ¿hay asociación con las declaraciones comerciales de la foto histórica? La segunda no prueba que podamos anticipar cierres al momento de ingreso.

Entrenamos un modelo común con este histórico entregado por el grupo. No fusionamos identidades entre empresas. La autorización de acceso por empresa será responsabilidad de la futura API/DB: un CSV local no implementa control de acceso.

In [ ]:
PERFILES = {
    'ingreso': ['canal'],
    'comercial': ['canal', 'modelo_sku', 'manifesto_cuota_inicial', 'forma_pago_declarada', 'cita_observada'],
}
def preparar_historico(df):
    x = df[['canal', 'modelo_sku', 'manifesto_cuota_inicial', 'forma_pago_declarada']].copy()
    x['cita_observada'] = np.where(df.pidio_cita.eq('SI'), 'SI', 'NO_OBSERVADA')
    return x.fillna('DESCONOCIDO').astype(str)

X = preparar_historico(historico)
assert not set(['numero_contactos', 'desenlace', 'empresa_id', 'horas_al_primer_contacto']) & set(X.columns)
display(X.head())

## 3. Entrenamiento, validación y prueba: tres usos distintos

- **Marzo–mayo:** ajustar los coeficientes o las divisiones del árbol.
- **Junio:** elegir modelo e hiperparámetros (las decisiones que controlan su complejidad).
- **Julio:** evaluar una vez el candidato elegido, después de reajustarlo con marzo–junio. No elegir otro ganador mirando julio.

Los límites se fijan por meses completos para no repartir un mismo día entre grupos. La codificación de categorías se ajusta dentro del `Pipeline` usando únicamente el grupo de entrenamiento correspondiente. Una categoría nueva se registra como incidencia y no rompe la ejecución.

**Limitación temporal:** separar por registro no demuestra que los cierres de mayo ya se conocieran en junio. Al faltar fecha del desenlace, esta sigue siendo una evaluación retrospectiva, no una simulación perfecta de producción.

In [ ]:
mascaras = {
    'entrenamiento': historico.fecha < pd.Timestamp('2026-06-01'),
    'validacion': historico.fecha.between('2026-06-01', '2026-06-30'),
    'prueba': historico.fecha >= pd.Timestamp('2026-07-01'),
}
partes = {k: historico[v].copy() for k, v in mascaras.items()}
assert sum(len(d) for d in partes.values()) == len(historico)
for d in partes.values(): assert d.y.nunique() == 2
assert partes['entrenamiento'].fecha.max() < partes['validacion'].fecha.min()
assert partes['validacion'].fecha.max() < partes['prueba'].fecha.min()
resumen_particion = pd.DataFrame([{'grupo': k, 'filas': len(d), 'cierres': int(d.y.sum()),
    'tasa_cierre': d.y.mean(), 'desde': d.fecha.min().strftime('%Y-%m-%d'),
    'hasta': d.fecha.max().strftime('%Y-%m-%d')} for k, d in partes.items()])
display(resumen_particion)

## 4. Cómo medir si ayuda a ordenar la cola

**Average precision (AP)** resume precisión y recuperación a distintos cortes. Más alto es mejor; una referencia sin señal es aproximadamente la tasa de cierre. Es la métrica principal para elegir en junio. No es la exactitud ni una probabilidad de compra.

También medimos el **primer 20% de la cola**, como escenario de capacidad, no como una capacidad real conocida:

- Precisión: qué proporción de ese grupo terminó comprando.
- Recuperación: qué proporción de todos los compradores quedó en ese grupo.
- Lift: precisión dividida por la tasa general; 1 equivale al rendimiento esperado de una selección aleatoria.
- ROC-AUC complementa la comparación; Brier evalúa las salidas numéricas de los modelos. No se calcula Brier para puntos de reglas ni FIFO porque no son probabilidades.

**Empates:** árboles y reglas pueden empatar muchas filas. Calculamos el resultado esperado al elegir aleatoriamente dentro del empate que cruza el corte; no usamos el orden del CSV para favorecerlos. Puede resultar un número fraccional de cierres esperados. FIFO desempata por ID solo para ser reproducible; las fechas históricas no tienen hora, así que el orden intradía real es desconocido.

La exactitud de predecir siempre «Perdido» sería alta por el desbalance, pero no ayudaría a encontrar compradores. Por eso no optimizamos accuracy, no duplicamos ejemplos con SMOTE y no balanceamos pesos por defecto: queremos probar primero una solución simple.

In [ ]:
def metrica_cola(y, score, fraccion=FRACCION_COLA):
    y, score = np.asarray(y), np.asarray(score, dtype=float)
    k = max(1, math.ceil(len(y) * fraccion))
    corte = np.sort(score)[-k]
    arriba, empate = score > corte, score == corte
    cierres = float(y[arriba].sum() + (k - arriba.sum()) * y[empate].mean())
    precision = cierres / k
    return {'k': k, 'cierres_esperados_top': cierres, 'precision_top20': precision,
            'recall_top20': cierres / y.sum() if y.sum() else np.nan,
            'lift_top20': precision / y.mean() if y.mean() else np.nan}

def evaluar(y, score, probabilidad=False):
    resultado = {'AP': average_precision_score(y, score),
                 'ROC_AUC': roc_auc_score(y, score) if len(set(y)) == 2 else np.nan,
                 **metrica_cola(y, score)}
    resultado['Brier'] = brier_score_loss(y, score) if probabilidad else np.nan
    return resultado

def score_reglas(x):
    # Puntos de evidencia, no dinero ni porcentajes de compra.
    modelo = ~x.modelo_sku.isin(['DESCONOCIDO', 'AMBIGUO', ''])
    inicial = x.manifesto_cuota_inicial.eq('SI')
    pago = x.forma_pago_declarada.isin(['credito', 'contado'])
    cita = x.cita_observada.eq('SI')
    return (10 * modelo.astype(int) + 30 * inicial.astype(int)
            + 10 * pago.astype(int) + 40 * cita.astype(int)).to_numpy() / 90

def score_fifo(df):
    orden = df.sort_values(['fecha', 'lead_id']).index
    return pd.Series(np.arange(len(df), 0, -1), index=orden).reindex(df.index).to_numpy()

## 5. Entrenar modelos pequeños y comparables

**Regresión logística:** suma contribuciones de las características y transforma el resultado a un valor entre 0 y 1. La regularización limita coeficientes extremos; probamos `C=0.1, 1, 10` (menor C, mayor regularización).

**Árbol:** aprende preguntas del tipo «¿hay solicitud de cita?». Probamos profundidad 2 y 3, con al menos 40 ejemplos en cada hoja. Eso limita reglas basadas en pocos casos. Una hoja informa la proporción histórica observada, no una garantía.

Todas las variables aquí son categóricas: `OneHotEncoder` crea indicadores por categoría. No escalamos importes porque no los usamos. La cuadrícula es pequeña para limitar la búsqueda con pocos cierres. En igualdad de AP preferimos el primer candidato de la lista, que empieza por modelos más simples.

Gemini ya es el modelo preentrenado de extracción; este entrenamiento tabular es local y no tiene costo de API. No necesitamos un segundo modelo de lenguaje para aprender estos pesos.

In [ ]:
def construir_modelo(tipo, parametro):
    estimador = (LogisticRegression(C=parametro, max_iter=2000, random_state=SEED)
                 if tipo == 'logistica' else DecisionTreeClassifier(
                     max_depth=parametro, min_samples_leaf=40, random_state=SEED))
    return Pipeline([('categorias', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                     ('modelo', estimador)])

candidatos, resultados_validacion = {}, []
train, val, test = (partes[k] for k in ['entrenamiento', 'validacion', 'prueba'])
for perfil, columnas in PERFILES.items():
    for tipo, parametros in [('logistica', [0.1, 1.0, 10.0]), ('arbol', [2, 3])]:
        for parametro in parametros:
            nombre = f'{perfil}_{tipo}_{parametro}'
            modelo = construir_modelo(tipo, parametro)
            modelo.fit(X.loc[train.index, columnas], train.y)
            p = modelo.predict_proba(X.loc[val.index, columnas])[:, 1]
            candidatos[nombre] = {'modelo': modelo, 'perfil': perfil, 'columnas': columnas}
            resultados_validacion.append({'candidato': nombre, 'perfil': perfil, 'tipo': tipo,
                'parametro': parametro, **evaluar(val.y, p, True)})

tabla_validacion = pd.DataFrame(resultados_validacion).sort_values('AP', ascending=False, kind='stable')
elegido = tabla_validacion.iloc[0].candidato
config_elegida = candidatos[elegido]
display(tabla_validacion.round(4))
display(pd.DataFrame([
    {'referencia': 'reglas', **evaluar(val.y, score_reglas(X.loc[val.index]))},
    {'referencia': 'FIFO', **evaluar(val.y, score_fifo(val))},
    {'referencia': 'constante', **evaluar(val.y, np.repeat(train.y.mean(), len(val)), True)},
]).round(4))
print('Configuración elegida usando solamente junio:', elegido)

## 6. Congelar la elección y abrir julio

Reajustamos **la configuración ya elegida** con marzo–junio. Este es el modelo que guardaremos: julio no entra en su entrenamiento. Así, las métricas de prueba corresponden exactamente al artefacto exportado. Un reajuste futuro con todos los datos necesitará un nuevo periodo de evaluación.

Antes de evaluar definimos una comprobación exigente: mejora de AP frente a reglas y de precisión en el primer 20% frente a FIFO, con intervalos bootstrap pareados cuyo límite inferior sea mayor que cero, además de lift mayor que 1. Remuestreamos las mismas filas para ambos métodos, 1.000 veces y con semilla fija. Los intervalos son orientativos: hay pocos positivos y no capturan toda la variación entre meses ni entre personas no identificables.

Incluso si pasa esa comprobación estadística, **no se promueve automáticamente a producción**: faltan fechas de captura de variables/desenlace y validación prospectiva. Conservamos un score experimental y una prioridad operativa de reglas separada, sin llamarla probabilidad.

In [ ]:
columnas_elegidas = config_elegida['columnas']
desarrollo = pd.concat([train, val])
modelo_final = clone(config_elegida['modelo']).fit(X.loc[desarrollo.index, columnas_elegidas], desarrollo.y)
pred_test = modelo_final.predict_proba(X.loc[test.index, columnas_elegidas])[:, 1]
reglas_test = score_reglas(X.loc[test.index])
fifo_test = score_fifo(test)
tabla_prueba = pd.DataFrame([
    {'metodo': elegido, **evaluar(test.y, pred_test, True)},
    {'metodo': 'reglas', **evaluar(test.y, reglas_test)},
    {'metodo': 'FIFO', **evaluar(test.y, fifo_test)},
    {'metodo': 'constante', **evaluar(test.y, np.repeat(desarrollo.y.mean(), len(test)), True)},
])
display(tabla_prueba.round(4))

rng = np.random.default_rng(SEED)
y_test = test.y.to_numpy()
diferencias = []
for _ in range(1000):
    ix = rng.integers(0, len(test), len(test))
    if len(np.unique(y_test[ix])) < 2: continue
    diferencias.append([
        average_precision_score(y_test[ix], pred_test[ix]) - average_precision_score(y_test[ix], reglas_test[ix]),
        metrica_cola(y_test[ix], pred_test[ix])['precision_top20'] - metrica_cola(y_test[ix], fifo_test[ix])['precision_top20'],
    ])
intervalos = pd.DataFrame(np.quantile(diferencias, [0.025, 0.5, 0.975], axis=0).T,
    columns=['limite_inferior_95', 'mediana', 'limite_superior_95'],
    index=['AP modelo menos reglas', 'precision_top20 modelo menos FIFO'])
mejora_estadistica = bool((intervalos.limite_inferior_95 > 0).all() and tabla_prueba.iloc[0].lift_top20 > 1)
modelo_apto_produccion = False  # Faltan fechas de observación y validación prospectiva, aunque mejoren métricas.
display(intervalos.round(4))
display(Markdown(f'**Lectura:** el candidato `{elegido}` obtiene AP **{tabla_prueba.iloc[0].AP:.3f}**, '
    f'con una tasa base de **{test.y.mean():.1%}**. Su lift en el primer 20% es **{tabla_prueba.iloc[0].lift_top20:.2f}**. '
    f'La comprobación estadística de mejora es **{"positiva" if mejora_estadistica else "no concluyente"}**. '
    'Se conserva como experimento; la prioridad exportada se basa en reglas visibles.'))

por_empresa = []
for empresa, g in test.assign(score=pred_test, reglas=reglas_test).groupby('empresa_id'):
    for nombre, scores, es_prob in [('modelo', g.score, True), ('reglas', g.reglas, False), ('FIFO', score_fifo(g), False)]:
        por_empresa.append({'empresa_id': empresa, 'metodo': nombre, 'filas': len(g), 'cierres': int(g.y.sum()),
                            **evaluar(g.y, scores, es_prob)})
display(pd.DataFrame(por_empresa).round(4))
display(Markdown('**Por qué mirar empresa por empresa:** la cola real se ordena dentro de cada empresa. '
    'El top 20% global es una comparación agregada y no equivale a repartir capacidad por empresa. '
    'La tabla anterior repite la comparación dentro de cada una; sus muestras son pequeñas y no justifican '
    'entrenar tres modelos separados. Un promedio global puede ocultar rendimiento inferior al azar en una empresa.'))

### Ver los resultados en gráficos

La curva de precisión/recuperación muestra el intercambio entre cubrir más compradores y concentrarlos en menos llamadas. La línea horizontal es la tasa general de cierre. El segundo gráfico compara cuántos cierres esperamos capturar con la misma capacidad del 20%. No son ventas adicionales causadas por el modelo: son cierres históricos reordenados.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for nombre, scores in [('Modelo', pred_test), ('Reglas', reglas_test)]:
    precision, recall, _ = precision_recall_curve(test.y, scores)
    axes[0].step(recall, precision, where='post', label=nombre)
axes[0].axhline(test.y.mean(), color='gray', linestyle='--', label='Tasa base')
axes[0].set(xlabel='Proporción de compradores recuperados', ylabel='Precisión', title='Prueba de julio: precisión y recuperación')
axes[0].legend()
barras = tabla_prueba.set_index('metodo').cierres_esperados_top.copy()
barras.index = ['Modelo', 'Reglas', 'FIFO', 'Aleatorio esperado']
barras.plot.bar(ax=axes[1], color=['#325ca8', '#249181', '#9c7937', '#999999'], rot=15)
axes[1].set(ylabel='Cierres esperados', xlabel='', title=f'Primer 20%: {math.ceil(len(test) * FRACCION_COLA)} leads')
fig.tight_layout()
fig.savefig(OUT / 'evaluacion_julio.png', dpi=160, bbox_inches='tight')
plt.show()

## 7. Entender qué aprendió cada familia

Para estudiar ambos enfoques, reajustamos con marzo–junio la mejor regresión y el mejor árbol de junio. **No los reevaluamos en julio para cambiar el ganador.**

En regresión, un coeficiente positivo aumenta el resultado matemático manteniendo las otras entradas fijas. No demuestra causalidad ni que el dato sea deseable; un coeficiente negativo tampoco justifica descartar a una persona. Con todas las categorías codificadas, son contribuciones al logit respecto al intercepto, no comparaciones contra una categoría de referencia omitida.

En el árbol podemos seguir cada pregunta hasta una hoja. `NO_INFORMA` significa falta de información, nunca una negativa inventada.

In [ ]:
modelos_explicacion = {}
for tipo in ['logistica', 'arbol']:
    mejor = tabla_validacion[tabla_validacion.tipo.eq(tipo)].iloc[0].candidato
    cfg = candidatos[mejor]
    modelos_explicacion[tipo] = (clone(cfg['modelo']).fit(X.loc[desarrollo.index, cfg['columnas']], desarrollo.y), cfg['columnas'])
logistica, cols_logistica = modelos_explicacion['logistica']
coeficientes = pd.DataFrame({'caracteristica': logistica['categorias'].get_feature_names_out(cols_logistica),
                            'coeficiente': logistica['modelo'].coef_[0]}).sort_values('coeficiente')
display(coeficientes)
arbol, cols_arbol = modelos_explicacion['arbol']
reglas_arbol = export_text(arbol['modelo'], feature_names=list(arbol['categorias'].get_feature_names_out(cols_arbol)), show_weights=True)
print(reglas_arbol)
fig, ax = plt.subplots(figsize=(15, 6))
plot_tree(arbol['modelo'], feature_names=arbol['categorias'].get_feature_names_out(cols_arbol),
          class_names=['Perdido', 'Cerrado'], filled=True, rounded=True, fontsize=8, ax=ax)
ax.set_title('Árbol elegido en validación: ejemplos por clase, no probabilidades garantizadas')
fig.tight_layout()
fig.savefig(OUT / 'arbol_explicado.png', dpi=160, bbox_inches='tight')
plt.show()

## 8. Prioridad operativa: una política visible mientras validamos el modelo

No ocultamos el resultado del entrenamiento ni inventamos que las reglas aumentarán conversiones. Esta política ordena **evidencia comercial observada**:

| Señal afirmativa | Puntos | Justificación de negocio |
|---|---:|---|
| Modelo identificado | 10 | Hay un producto concreto del que hablar. |
| Inicial positiva mencionada | 30 | Hay un monto inicial declarado; no acredita solvencia ni aprobación. |
| Forma de pago declarada | 10 | Sabemos qué información comercial preparar; crédito y contado valen igual. |
| Cliente pidió cita | 40 | Hay una solicitud de atención concreta. Una oferta del asesor no cuenta. |

`score_prioridad = 100 × puntos / 90`. Los pesos son una hipótesis comercial explícita, **no aprendida**. La comparación histórica de reglas usa exactamente esta fórmula. No sumamos cotización, objeciones o intención porque no tienen equivalencia histórica; se conservan como contexto para el asesor.

**Temperatura operativa (no probabilidad de cierre):** caliente si pidió cita; tibio si declaró inicial positiva o forma de pago; sin información suficiente en los demás casos. No llamamos «frío» a quien no tuvo oportunidad de responder. Los faltantes aportan cero puntos, pero no se convierten en hechos negativos ni generan eliminación.

La cola se separa por empresa. Dentro de cada una, ordenamos por score y, en empate, antigüedad e ID. Las consultas descartadas del CRM se conservan para revisión fuera de la cola activa. Para evitar que quienes no tienen conversación queden olvidados, dejamos una **cola de primer contacto** separada; no proponemos que el score sea el único criterio de atención.

## 9. Aplicar a los leads actuales sin mezclar empresas ni inventar datos

- Una salida por `(empresa_id, lead_consolidado_id)`; nunca agrupamos por teléfono entre empresas.
- Leemos solo las conversaciones utilizables del notebook 03. Las descartadas no entran ni se recuperan aquí.
- Cuando hay varias conversaciones utilizables, elegimos la más reciente con fecha válida como **foto de trabajo**, conservando todos sus IDs. No mezclamos afirmaciones de distintas fechas. Esto no afirma que un dato antiguo haya dejado de ser cierto.
- Si hay empate en la fecha más reciente, la hora es desconocida entre conversaciones o hay fechas inválidas, no elegimos arbitrariamente: dejamos las señales conversacionales desconocidas y registramos el caso para revisión.
- Canal y modelo proceden de la consulta vinculada a esa conversación; sin conversación, usamos la consulta más reciente. Si no se puede decidir su orden, solo conservamos valores iguales entre las consultas candidatas.
- SKU: preferimos el único modelo reconocido en esa conversación; si menciona varios, queda ambiguo. Se conservan todos los modelos en la extracción original. No escogemos el más caro ni inferimos preferencia.
- Inicial: positiva → SI; cero explícito → NO; ausente → NO_INFORMA. Rangos solo positivos también permiten SI. Nunca usamos el monto como probabilidad ni inventamos moneda.
- Cita: solo `cliente_solicito=si` cuenta como solicitud observada. Aceptar una cita ofrecida no se transforma en haberla pedido. Cotizaciones y ofertas de crédito quedan disponibles en el contexto.

Las conversaciones sin vínculo o empresa permanecen aparte con referencias y motivo. No podemos adjudicarlas a un cliente ni a una comercializadora por intuición.

In [ ]:
conv_por_id = {c['conversacion_id']: c for c in conversaciones}
assert len(conv_por_id) == len(conversaciones)
assert {e['extraccion']['conversacion_id'] for e in extracciones} == set(conv_por_id)
consultas_por_id = consultas.set_index('lead_id', drop=False)
assert consultas_por_id.index.is_unique
claves_lead = set(zip(leads.empresa_id, leads.lead_consolidado_id))
por_lead, huerfanas = {}, []
for e in extracciones:
    clave = (e['empresa_id'], e['lead_consolidado_id'])
    cid = e['extraccion']['conversacion_id']
    if clave not in claves_lead:
        huerfanas.append({'conversacion_id': cid, 'empresa_id': e['empresa_id'], 'lead_id': e['lead_id'],
                         'motivo': 'Sin vínculo verificable a un lead y empresa; conservar fuera de la cola.'})
        continue
    assert e['lead_id'] in consultas_por_id.index
    q = consultas_por_id.loc[e['lead_id']]
    assert (q.empresa_id, q.lead_consolidado_id) == clave
    c = conv_por_id[cid]
    assert (c['empresa_id'], c['lead_consolidado_id'], c['lead_id']) == (*clave, e['lead_id'])
    por_lead.setdefault(clave, []).append(e)
incidencias.extend({'tipo': 'conversacion_sin_vinculo', **r} for r in huerfanas)

def elegir_ultima(fechas):
    # Comparar por día si alguna fuente no informa hora: no fabricar medianoche real.
    parsed = pd.to_datetime(pd.Series(fechas), format='mixed', errors='coerce')
    if parsed.isna().any(): return None, 'fecha_invalida'
    dia = parsed.dt.normalize()
    candidatos = np.flatnonzero(dia.eq(dia.max()))
    if len(candidatos) == 1: return int(candidatos[0]), None
    if any(len(str(fechas[i])) <= 10 for i in candidatos): return None, 'orden_intradia_desconocido'
    ix = np.flatnonzero(parsed.eq(parsed.max()))
    return (int(ix[0]), None) if len(ix) == 1 else (None, 'empate_fecha')

def inicial_de_extraccion(ex):
    d = ex['dinero']['cuota_inicial']
    if not d['evidencias']: return 'NO_INFORMA'
    if d['valor'] is not None:
        return 'SI' if d['valor'] > 0 else ('NO' if d['valor'] == 0 else 'NO_INFORMA')
    if d['minimo'] is not None and d['minimo'] > 0: return 'SI'
    return 'NO_INFORMA'

### Construir una fila por lead y conservar su contexto

El bucle siguiente aplica las reglas de vínculo y fecha anteriores. Las señales comerciales salen de una sola conversación verificable; los demás IDs se conservan para consulta.

In [ ]:
filas, contextos = [], []
for _, lead in leads.iterrows():
    clave = (lead.empresa_id, lead.lead_consolidado_id)
    qs = consultas[(consultas.empresa_id == clave[0]) & (consultas.lead_consolidado_id == clave[1])]
    assert len(qs) > 0
    es = por_lead.get(clave, [])
    e, problema = None, None
    if es:
        posicion, problema = elegir_ultima([conv_por_id[x['extraccion']['conversacion_id']]['fecha_inicio'] for x in es])
        if posicion is not None: e = es[posicion]
        else: incidencias.append({'tipo': 'orden_conversaciones_ambiguo', 'empresa_id': clave[0],
            'lead_consolidado_id': clave[1], 'motivo': problema, 'accion': 'No usar señales conversacionales; conservar fuentes.'})
    qpos, qproblema = elegir_ultima(qs.fecha_registro.tolist())
    q = consultas_por_id.loc[e['lead_id']] if e else (qs.iloc[qpos] if qpos is not None else None)
    if qproblema and e is None:
        incidencias.append({'tipo': 'orden_consultas_ambiguo', 'empresa_id': clave[0],
            'lead_consolidado_id': clave[1], 'motivo': qproblema, 'accion': 'Solo conservar canal/modelo comunes; no inventar orden.'})
    def valor_consulta(c):
        if q is not None: return str(q[c]) if pd.notna(q[c]) else 'DESCONOCIDO'
        valores = qs[c].fillna('DESCONOCIDO').unique()
        return str(valores[0]) if len(valores) == 1 else 'DESCONOCIDO'
    ex = e['extraccion'] if e else None
    sku = valor_consulta('modelo_sku')
    if e and ex['modelos_interes']:
        skus = {m.get('sku') for m in e['catalogo'] if m.get('sku')}
        sku = next(iter(skus)) if len(skus) == 1 and all(m.get('sku') for m in e['catalogo']) else 'AMBIGUO'
    pago = ex['forma_pago']['valor'] if ex else 'desconocida'
    pago = pago if pago in ['credito', 'contado'] else 'no_informa'
    fila = {'empresa_id': clave[0], 'lead_consolidado_id': clave[1],
        'lead_id_contexto': str(q.lead_id) if q is not None else None,
        'conversacion_id_contexto': ex['conversacion_id'] if ex else None,
        'numero_conversaciones_utilizables': len(es),
        'canal': valor_consulta('canal'), 'modelo_sku': sku,
        'manifesto_cuota_inicial': inicial_de_extraccion(ex) if ex else 'NO_INFORMA',
        'forma_pago_declarada': pago,
        'cita_observada': 'SI' if ex and ex['cita']['cliente_solicito']['estado'] == 'si' else 'NO_OBSERVADA',
        'fecha_antiguedad': lead.primera_fecha_registro_resuelta,
        'todas_consultas_descartadas': bool(qs.estado_gestion.eq('Descartado').all()),
        'estado_contexto': 'conversacion_utilizable' if e else ('revision_orden' if problema else 'sin_conversacion_utilizable')}
    filas.append(fila)
    contextos.append({'empresa_id': clave[0], 'lead_consolidado_id': clave[1],
        'conversaciones_utilizables': [x['extraccion']['conversacion_id'] for x in es],
        'conversacion_id_contexto': fila['conversacion_id_contexto'],
        'extraccion_contexto': ex, 'criterio': 'Foto más reciente verificable; no acumular declaraciones de otras fechas.'})



### Calcular los dos puntajes y ordenar las colas

`score_prioridad` es el resultado de reglas operativas. `score_modelo_experimental` es la salida matemática del modelo multiplicada por 100 para inspección; no ordena la cola. Las posiciones empiezan en 1 dentro de cada empresa y cada tipo de cola.

In [ ]:
actuales = pd.DataFrame(filas)
X_actual = actuales[PERFILES['comercial']].fillna('DESCONOCIDO').astype(str)
actuales['score_prioridad'] = np.round(100 * score_reglas(X_actual), 2)
actuales['score_modelo_experimental'] = np.round(100 * modelo_final.predict_proba(X_actual[columnas_elegidas])[:, 1], 4)
actuales['temperatura'] = np.select([
    actuales.cita_observada.eq('SI'),
    actuales.manifesto_cuota_inicial.eq('SI') | actuales.forma_pago_declarada.isin(['credito', 'contado'])],
    ['caliente', 'tibio'], default='sin_informacion_suficiente')
actuales['cola'] = np.select([actuales.todas_consultas_descartadas,
    actuales.estado_contexto.eq('revision_orden'), actuales.temperatura.eq('sin_informacion_suficiente')],
    ['revision_descartado_crm', 'revision_datos', 'primer_contacto_o_ampliar_informacion'], default='seguimiento_comercial')

def explicar_reglas(r):
    motivos = []
    if r.modelo_sku not in ['DESCONOCIDO', 'AMBIGUO', '']: motivos.append('Modelo identificado: +10 puntos.')
    if r.manifesto_cuota_inicial == 'SI': motivos.append('Inicial positiva declarada: +30 puntos.')
    if r.forma_pago_declarada in ['credito', 'contado']: motivos.append('Forma de pago declarada: +10 puntos.')
    if r.cita_observada == 'SI': motivos.append('Cliente pidió cita: +40 puntos.')
    return ' '.join(motivos) + ' Escala: puntos/90*100; no es probabilidad. Ausencia de señal no es rechazo.'
actuales['explicacion_prioridad'] = actuales.apply(explicar_reglas, axis=1)
actuales['accion_sugerida'] = actuales.cola.map({
    'revision_descartado_crm': 'Revisar motivo del descarte en CRM antes de reactivar.',
    'revision_datos': 'Revisar orden temporal antes de usar declaraciones.',
    'primer_contacto_o_ampliar_informacion': 'Confirmar interés y completar información; no asumir desinterés.',
    'seguimiento_comercial': 'Retomar la conversación y responder a lo solicitado por el cliente.'})
actuales.loc[actuales.cita_observada.eq('SI') & actuales.cola.eq('seguimiento_comercial'), 'accion_sugerida'] = 'Confirmar disponibilidad y estado de la cita solicitada; no asumir que sigue pendiente.'
actuales['metodo_prioridad'] = 'reglas_evidencia_v1'
actuales['modelo_experimental'] = elegido
actuales['modelo_apto_produccion'] = False
actuales['_fecha_orden'] = pd.to_datetime(actuales.fecha_antiguedad, format='mixed', errors='coerce')
actuales = actuales.sort_values(['empresa_id', 'cola', 'score_prioridad', '_fecha_orden', 'lead_consolidado_id'],
    ascending=[True, True, False, True, True], na_position='last')
actuales['posicion_en_cola_empresa'] = actuales.groupby(['empresa_id', 'cola']).cumcount() + 1
actuales = actuales.drop(columns='_fecha_orden')
display(pd.crosstab(actuales.empresa_id, actuales.temperatura))
display(actuales[['empresa_id', 'lead_consolidado_id', 'score_prioridad', 'temperatura', 'cola', 'explicacion_prioridad']].head(12))

## 10. Cobertura, cambios de distribución y explicación individual del modelo

Los actuales y el histórico no son la misma población: en los actuales usamos extracción de conversaciones filtradas, mientras el histórico contiene campos de CRM de una foto final. Mostramos sus distribuciones, categorías nuevas y limitaciones. Una categoría desconocida se codifica como todos ceros para ese campo; esto permite calcular, pero **no demuestra que la predicción sea fiable**. Por eso se registra y permanece experimental.

La explicación individual de regresión guarda intercepto y contribuciones al logit. La del árbol guarda las condiciones de la ruta y los ejemplos de la hoja. Son explicaciones del cálculo del modelo, separadas de los puntos que usa la cola operativa.

In [ ]:
distribuciones = []
for campo in PERFILES['comercial']:
    a = X[campo].value_counts(normalize=True)
    b = X_actual[campo].value_counts(normalize=True)
    for valor in sorted(set(a.index) | set(b.index)):
        distribuciones.append({'campo': campo, 'valor': valor, 'historico_proporcion': a.get(valor, 0), 'actual_proporcion': b.get(valor, 0)})
    nuevas = set(X_actual[campo]) - set(X.loc[desarrollo.index, campo])
    for i in X_actual.index[X_actual[campo].isin(nuevas)]:
        incidencias.append({'tipo': 'categoria_no_vista', 'empresa_id': filas[i]['empresa_id'],
            'lead_consolidado_id': filas[i]['lead_consolidado_id'], 'campo': campo,
            'valor': X_actual.loc[i, campo], 'accion': 'Conservar; score ML experimental, sin evidencia de generalización.'})
distribuciones = pd.DataFrame(distribuciones)
display(distribuciones.round(3))

transformados = modelo_final['categorias'].transform(X_actual[columnas_elegidas])
nombres = modelo_final['categorias'].get_feature_names_out(columnas_elegidas)
estimador = modelo_final['modelo']
explicaciones_modelo = []
for i, valores in enumerate(transformados):
    if isinstance(estimador, LogisticRegression):
        contrib = valores * estimador.coef_[0]
        detalle = {'tipo': 'contribuciones_logit', 'intercepto': float(estimador.intercept_[0]),
            'contribuciones': [{'caracteristica': str(nombres[j]), 'aporte': float(contrib[j])}
                              for j in np.flatnonzero(valores)],
            'logit_total': float(estimador.intercept_[0] + contrib.sum())}
    else:
        nodos = estimador.decision_path(valores.reshape(1, -1)).indices
        condiciones = []
        for nodo in nodos:
            f = estimador.tree_.feature[nodo]
            if f >= 0:
                umbral = estimador.tree_.threshold[nodo]
                condiciones.append({'caracteristica': str(nombres[f]), 'valor': float(valores[f]),
                    'operador': '<=' if valores[f] <= umbral else '>', 'umbral': float(umbral)})
        hoja = int(estimador.apply(valores.reshape(1, -1))[0])
        detalle = {'tipo': 'ruta_arbol', 'condiciones': condiciones, 'hoja': hoja,
                   'ejemplos_entrenamiento_hoja': int(estimador.tree_.n_node_samples[hoja])}
    explicaciones_modelo.append({'empresa_id': filas[i]['empresa_id'],
        'lead_consolidado_id': filas[i]['lead_consolidado_id'], 'modelo': elegido, **detalle})
display(pd.DataFrame(explicaciones_modelo).head(3))

## 11. Controles antes de exportar

Comprobamos que no se perdió ningún lead, que cada conversación usada mantiene la empresa y el lead originales, que no se reutilizaron las conversaciones excluidas, que el orden no depende de mezclar empresas y que las fuentes anteriores permanecen idénticas. También probamos casos de negocio: oferta del asesor ≠ solicitud del cliente; ausencia de inicial ≠ inicial cero; misma persona en otra empresa ≠ identidad fusionada.

El score es reproducible con estas fuentes y esta versión. Los archivos generados en `processed` son una etapa intermedia; no reemplazan la base de datos ni la API con aislamiento por empresa exigidas en los pasos siguientes.

In [ ]:
assert len(actuales) == len(leads)
assert set(zip(actuales.empresa_id, actuales.lead_consolidado_id)) == claves_lead
assert not actuales.duplicated(['empresa_id', 'lead_consolidado_id']).any()
assert actuales.score_prioridad.between(0, 100).all()
assert actuales.score_modelo_experimental.between(0, 100).all()
assert set(actuales.conversacion_id_contexto.dropna()) <= set(conv_por_id)
assert all(sha(DATA / p) == huella for p, huella in hashes_entrada.items())
for _, g in actuales.groupby(['empresa_id', 'cola']):
    assert g.score_prioridad.is_monotonic_decreasing
    assert g.posicion_en_cola_empresa.tolist() == list(range(1, len(g) + 1))
caso_sin_datos = pd.DataFrame([{'modelo_sku': 'DESCONOCIDO', 'manifesto_cuota_inicial': 'NO_INFORMA',
    'forma_pago_declarada': 'no_informa', 'cita_observada': 'NO_OBSERVADA'}])
assert score_reglas(caso_sin_datos)[0] == 0
assert np.isclose(score_reglas(caso_sin_datos.assign(cita_observada='SI'))[0], 40/90)
assert inicial_de_extraccion({'dinero': {'cuota_inicial': {'valor': None, 'minimo': None, 'evidencias': []}}}) == 'NO_INFORMA'
assert inicial_de_extraccion({'dinero': {'cuota_inicial': {'valor': 0, 'minimo': None, 'evidencias': [{'texto': 'No tengo inicial'}]}}}) == 'NO'
assert elegir_ultima(['2026-08-01', '2026-08-01 10:00:00'])[0] is None
assert elegir_ultima(['2026-08-01', '2026-08-02 10:00:00'])[0] == 1
print('Controles de cobertura, identidad, orden, faltantes, límites y fuentes: correctos.')

## 12. Exportar y dejar trazabilidad para los siguientes pasos

### Resultados para integrar

- `data/processed/priorizacion_leads.csv` y `.json`: una fila por lead/empresa, score de reglas, temperatura, cola, explicación, acción y score ML **experimental** separado.
- `contexto_priorizacion.json`: extracción elegida y referencias de todas las conversaciones utilizables del lead.
- `incidencias_modelado.json`: exclusiones de entrenamiento, vínculos desconocidos, orden ambiguo, categorías nuevas y limitaciones generales. Ninguna exclusión elimina la fuente.
- `manifest_clasificacion.json`: huellas de entradas/salidas, versión, decisión y conteos.

### Material de estudio en `outputs/clasificacion_04`

Métricas de validación/prueba, predicciones de julio con verdad observada, intervalos bootstrap, coeficientes, árbol, gráficos, distribución de variables, explicaciones individuales y ficha del modelo. El archivo `modelo_experimental.joblib` contiene el pipeline y las columnas requeridas; cargar únicamente este artefacto local de confianza.

**Qué falta para defender un modelo como predictor operativo:** capturar las variables a una hora de corte fija; registrar fecha de desenlace y horizonte de compra; conservar casos sin gestión como censurados hasta conocer su evolución; recopilar cierres posteriores; comprobar rendimiento por empresa y calibración. No ajustamos probabilidades con los pocos cierres de julio: consumiría el conjunto reservado y ocultaría el problema de datos.

In [ ]:
def guardar_json(path, contenido):
    Path(path).write_text(json.dumps(contenido, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
def registros(df): return json.loads(df.to_json(orient='records', force_ascii=False))

limitaciones = [
    'Datos sintéticos; desempeño no validado en clientes reales.',
    'Sin fecha de desenlace ni de captura de señales: evaluación temporal retrospectiva, no prospectiva.',
    'Entrenamiento condicionado a leads gestionados; Sin gestión no equivale a Perdido.',
    'Modelo cotizado histórico e interés actual pueden tener distinto significado.',
    'No hay identificadores personales históricos para detectar repetición de personas entre particiones.',
    'Solo se usan conversaciones aceptadas; el filtrado y los faltantes cambian la población observada.',
    'Puntajes ML sin calibración validada; no son porcentajes de compra garantizados.',
    'Pesos y temperaturas de reglas son decisiones comerciales provisionales, no efectos causales.',
    'La separación por empresa de archivos y colas no implementa autorización de acceso de una aplicación.',
]
incidencias.extend({'tipo': 'limitacion_general', 'motivo': s, 'accion': 'Documentar; mantener modelo experimental.'} for s in limitaciones)
actuales.to_csv(DATA / 'priorizacion_leads.csv', index=False, encoding='utf-8')
guardar_json(DATA / 'priorizacion_leads.json', registros(actuales))
guardar_json(DATA / 'contexto_priorizacion.json', contextos)
guardar_json(DATA / 'incidencias_modelado.json', {'version': VERSION, 'incidencias': incidencias})
tabla_validacion.to_csv(OUT / 'metricas_validacion.csv', index=False)
tabla_prueba.to_csv(OUT / 'metricas_prueba.csv', index=False)
pd.DataFrame(por_empresa).to_csv(OUT / 'metricas_prueba_empresa.csv', index=False)
resumen_particion.to_csv(OUT / 'particiones.csv', index=False)
intervalos.to_csv(OUT / 'intervalos_bootstrap.csv')
coeficientes.to_csv(OUT / 'coeficientes_logistica.csv', index=False)
distribuciones.to_csv(OUT / 'distribuciones.csv', index=False)
(OUT / 'reglas_arbol.txt').write_text(reglas_arbol, encoding='utf-8')
test[['empresa_id', 'lead_id', 'fecha_registro', 'y']].assign(
    score_modelo=pred_test, score_reglas=reglas_test, orden_fifo=fifo_test).to_csv(OUT / 'predicciones_prueba.csv', index=False)
guardar_json(OUT / 'explicaciones_modelo.json', explicaciones_modelo)
joblib.dump({'pipeline': modelo_final, 'columnas': columnas_elegidas, 'version': VERSION,
             'candidato': elegido, 'uso': 'experimental_no_operativo'}, OUT / 'modelo_experimental.joblib')
cargado = joblib.load(OUT / 'modelo_experimental.joblib')
assert np.allclose(cargado['pipeline'].predict_proba(X.loc[test.index, cargado['columnas']])[:, 1], pred_test)
paquetes = {p: metadata.version(p) for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'joblib', 'nbformat', 'nbclient', 'ipykernel']}
ficha = {'version': VERSION, 'semilla': SEED, 'python': platform.python_version(), 'paquetes': paquetes,
    'objetivo': 'Cerrado frente a Perdido entre leads gestionados; horizonte no disponible.',
    'candidato_elegido': elegido, 'columnas': columnas_elegidas, 'seleccion': 'Mayor AP en junio; elección congelada antes de julio.',
    'entrenamiento_final': '2026-03-01 a 2026-06-30', 'prueba': '2026-07-01 a 2026-07-28',
    'particiones': registros(resumen_particion), 'metricas_prueba': registros(tabla_prueba),
    'bootstrap': registros(intervalos.reset_index(names='comparacion')),
    'mejora_estadistica': mejora_estadistica, 'modelo_apto_produccion': modelo_apto_produccion,
    'metodo_operativo': 'reglas_evidencia_v1', 'pesos_reglas': {'modelo': 10, 'inicial_positiva': 30, 'pago_declarado': 10, 'cita_solicitada': 40},
    'limitaciones': limitaciones, 'hashes_entrada': hashes_entrada}
guardar_json(OUT / 'ficha_modelo.json', ficha)
(OUT / 'README.md').write_text(
    '# Clasificación y priorización — paso 04\n\n'
    'Generado por `notebooks/04_clasificacion_priorizacion.ipynb`. Ejecutar el notebook completo con '
    '`requirements-notebook04.txt`; no requiere llamadas a Gemini.\n\n'
    f'Modelo seleccionado en validación: `{elegido}`. Se conserva como experimental. '
    'La prioridad operativa usa reglas documentadas, no probabilidades de compra.\n\n'
    'Consultar `ficha_modelo.json` para fechas, métricas, versiones, límites y hashes de entradas. '
    '`metricas_validacion.csv` contiene todos los candidatos; `metricas_prueba.csv` evalúa la elección congelada. '
    '`metricas_prueba_empresa.csv` compara modelo, reglas y FIFO dentro de cada empresa.\n\n'
    'El pipeline exportado se entrenó con marzo–junio; julio permanece fuera de entrenamiento. '
    '`modelo_experimental.joblib` requiere las columnas y versiones registradas en la ficha. '
    'Cargar solo artefactos locales de confianza.\n\n'
    'Las salidas para integrar están en `data/processed/priorizacion_leads.csv`, su versión JSON, '
    '`contexto_priorizacion.json`, `incidencias_modelado.json` y `manifest_clasificacion.json`. '
    'No sustituyen la base de datos final.\n', encoding='utf-8')
salidas = ['priorizacion_leads.csv', 'priorizacion_leads.json', 'contexto_priorizacion.json', 'incidencias_modelado.json']
manifest = {'version': VERSION, 'notebook': 'notebooks/04_clasificacion_priorizacion.ipynb',
    'leads_priorizados': len(actuales), 'conversaciones_utilizables': len(extracciones),
    'conversaciones_sin_vinculo': len(huerfanas), 'historicos_entrenables': len(historico),
    'historicos_fuera_entrenamiento': int((sin_etiqueta | sin_fecha).sum()),
    'temperaturas': actuales.temperatura.value_counts().to_dict(), 'colas': actuales.cola.value_counts().to_dict(),
    'modelo_experimental': elegido, 'modelo_apto_produccion': False,
    'metodo_prioridad': 'reglas_evidencia_v1', 'hashes_entrada': hashes_entrada,
    'hashes_salida': {p: sha(DATA / p) for p in salidas},
    'modelo_sha256': sha(OUT / 'modelo_experimental.joblib')}
guardar_json(DATA / 'manifest_clasificacion.json', manifest)
assert all(sha(DATA / p) == h for p, h in hashes_entrada.items())
display(Markdown(f'### Resultado de esta ejecución\n\n'
    f'- **{len(actuales)} leads** conservados y priorizados por empresa.\n'
    f'- **{len(historico)} ejemplos históricos** con desenlace observado; **{int(sin_etiqueta.sum())}** sin etiqueta apartados del entrenamiento.\n'
    f'- Modelo elegido en junio: **{elegido}**; guardado y verificado al recargar.\n'
    f'- Comprobación estadística de mejora: **{"positiva" if mejora_estadistica else "no concluyente"}**.\n'
    '- Modelo experimental; reglas operativas visibles, sin presentar puntos como probabilidad.\n'
    f'- **{len(huerfanas)} conversaciones** sin vínculo conservadas en incidencias.\n'
    '- Entradas anteriores intactas; exportaciones y hashes generados.'))

## 13. Guía para estudiar y sustentar

1. **¿Qué aprendió el modelo?** Asociaciones entre campos disponibles y el desenlace de leads gestionados. No quién comprará con certeza.
2. **¿Por qué no usar todos los campos?** Algunos ocurren después de priorizar; otros no existen de forma equivalente en los leads actuales. Más columnas pueden hacer una evaluación engañosa.
3. **¿Por qué tres grupos?** Entrenamiento aprende; validación elige; prueba mide una elección ya congelada. Mirar julio para volver a elegir contaminaría la prueba.
4. **¿Por qué un modelo sencillo?** Hay pocos cierres, queremos inspeccionar qué usa y comparar contra referencias simples. Complejidad adicional necesita evidencia.
5. **¿Por qué no desplegarlo automáticamente?** La utilidad estadística y la disponibilidad temporal de los datos son requisitos distintos. El histórico no permite resolver ambos.
6. **¿Cómo se justifica la prioridad entregada?** Cada punto depende de una señal concreta y se explica por lead. Los pesos son hipótesis de negocio visibles, pendientes de validación operativa.
7. **¿Se borraron personas con datos incompletos?** No. Se conservan con contexto desconocido o en cola de revisión. Las conversaciones rechazadas en el paso 03 siguen fuera.
8. **¿Se separaron las empresas?** Sí para identidad, vínculos y orden de las colas. La autorización de usuarios todavía corresponde al paso de API/DB.

### Fuentes técnicas

- [Regresión logística — scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [Árboles de decisión — scikit-learn](https://scikit-learn.org/stable/modules/tree.html)
- [Prevención de fuga de información y pipelines — scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html)

**Siguiente etapa del proyecto:** persistir resultados y trazabilidad en una base de datos, sin perder empresa, fuentes, versión del método ni distinción entre dato observado y score. El entrenamiento ya está hecho; antes de mejorar su complejidad debemos resolver la calidad temporal de las etiquetas.

## 14. Segunda iteración: compensar el desbalance

**Estado: ejecutado por el usuario y revisado; ver sección 14.8.** Las salidas de las secciones 1–13 pertenecen a la primera iteración sin pesos. Las de esta sección pertenecen a la comparación con pesos.

### Qué queremos comprobar

Hay muchos más leads perdidos que compradores. Un modelo podría favorecer a la mayoría. Ya usamos métricas apropiadas para detectar ese problema; ahora probaremos una intervención **durante el entrenamiento**: `class_weight="balanced"` frente a `class_weight=None`.

El peso de cada clase es `n / (2 × cantidad_de_ejemplos_de_esa_clase)`. En cada ajuste se calcula exclusivamente con las etiquetas de entrenamiento. Cada comprador tendrá aproximadamente nueve veces el peso de un perdido, según la proporción del periodo usado. Se pondera la pérdida de la regresión y el criterio de división del árbol; no se crean filas ni se borran perdidos.

**Hipótesis:** dar más peso a los cierres podría ayudar a encontrarlos. **Riesgo:** también puede aumentar los falsos positivos y alterar la escala de las salidas. No damos por hecho que mejore el orden de prioridad. Un valor de 0.70 después de ponderar no equivale automáticamente a un 70% de probabilidad real de compra.

No hacemos submuestreo, sobremuestreo ni SMOTE: queremos aislar el efecto de los pesos sin perder información ni generar combinaciones categóricas artificiales. Compensar el desbalance no corrige fugas de información, etiquetas inciertas ni falta de señal.

### Cómo ejecutar este bloque

Puedes ejecutar **solo las celdas nuevas desde esta sección, en orden**, incluso con el kernel recién reiniciado: incluye sus importaciones y carga de datos. También funciona al ejecutar todo el notebook, pero las secciones anteriores volverán a regenerar sus propios resultados.

Este bloque guarda sus resultados en `outputs/clasificacion_04/desbalance_v2/` cuando lo ejecutes. No reemplaza el modelo anterior ni modifica las prioridades de `data/processed`. La interpretación y la decisión final quedan pendientes para revisarlas juntos.

In [ ]:
# Preparación independiente: esta celda no depende del estado de las secciones anteriores.
from pathlib import Path
import json, hashlib, math, platform
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display, Markdown
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    precision_score, recall_score, confusion_matrix,
)

DB_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / 'data/processed/historico_cierres.csv').exists())
DB_FUENTE = DB_ROOT / 'data/processed/historico_cierres.csv'
DB_OUT = DB_ROOT / 'outputs/clasificacion_04/desbalance_v2'
DB_SEED = 42
DB_FRACCION = 0.20
DB_VERSION = '04-desbalance-v2'
db_hash_fuente = hashlib.sha256(DB_FUENTE.read_bytes()).hexdigest()
db_fuente = pd.read_csv(DB_FUENTE)
assert not db_fuente.duplicated(['empresa_id', 'lead_id']).any()
assert set(db_fuente.desenlace.dropna()) <= {'Cerrado', 'Perdido', 'Sin gestión'}
db_fuente['fecha'] = pd.to_datetime(db_fuente.fecha_registro, format='%Y-%m-%d', errors='coerce')
db_utilizable = db_fuente.desenlace.isin(['Cerrado', 'Perdido']) & db_fuente.fecha.notna()
db_datos = db_fuente.loc[db_utilizable].copy()
db_datos['y'] = db_datos.desenlace.eq('Cerrado').astype(int)
db_fuera = db_fuente.loc[~db_utilizable, ['empresa_id', 'lead_id', 'desenlace', 'fecha_registro']].copy()
db_fuera['motivo'] = np.where(
    ~db_fuente.loc[~db_utilizable, 'desenlace'].isin(['Cerrado', 'Perdido']),
    'Sin resultado observado: no etiquetar como perdido.',
    'Fecha inválida: no permite partición temporal.',
)
db_X = db_datos[['canal', 'modelo_sku', 'manifesto_cuota_inicial', 'forma_pago_declarada']].copy()
db_X['cita_observada'] = np.where(db_datos.pidio_cita.eq('SI'), 'SI', 'NO_OBSERVADA')
db_X = db_X.fillna('DESCONOCIDO').astype(str)
DB_PERFILES = {
    'ingreso': ['canal'],
    'comercial': ['canal', 'modelo_sku', 'manifesto_cuota_inicial', 'forma_pago_declarada', 'cita_observada'],
}
display(db_datos.groupby('desenlace').agg(registros=('y', 'size')).assign(
    proporcion=lambda d: d.registros / d.registros.sum()))
print(f'Registros fuera de entrenamiento, conservados en la fuente: {len(db_fuera)}')

### 14.1. Protocolo temporal fijado antes de ejecutar

Julio ya se examinó en la primera iteración. **No vuelve a ser una prueba completamente nueva** aunque entrenemos otro modelo. En esta segunda iteración:

| Ronda | Entrenar con | Validar con | Uso |
|---|---|---|---|
| 1 | Marzo–abril | Mayo | Medir candidatos sin aprender de mayo. |
| 2 | Marzo–mayo | Junio | Medir los mismos candidatos un mes después. |
| Ajuste final | Marzo–junio | — | Reajustar los dos representantes elegidos. |
| Comprobación adicional | — | Julio | Describir resultados; no volver a elegir ni ajustar. |

Mayo y junio **también son datos explorados**: esta validación ayuda a comparar de forma consistente, no elimina la adaptación a datos conocidos. Una evaluación prospectiva definitiva necesitará nuevos cierres.

Usamos la misma lista de variables y el mismo presupuesto de búsqueda para ambos regímenes. Elegimos un representante sin pesos y otro balanceado por **AP media de las dos rondas**, dando igual importancia a cada mes. En empate usamos el orden de candidatos definido en el código, empezando por ingreso, regresión con C menor y árbol menos profundo. No usamos julio para elegir.

Además mostramos diferencias **con exactamente el mismo perfil, algoritmo e hiperparámetro**. Esta comparación pareada permite aislar el efecto de los pesos; comparar solo dos ganadores diferentes no lo permite.

Los grupos de validación y julio conservan sus filas y proporciones originales. No se ponderan sus métricas. El histórico comercial conserva las limitaciones temporales y semánticas de la primera iteración.

In [ ]:
DB_RONDAS = [
    ('validar_mayo', '2026-05-01', '2026-06-01'),
    ('validar_junio', '2026-06-01', '2026-07-01'),
]
db_rondas, db_pesos, db_particiones = [], [], []
for nombre, inicio, fin in DB_RONDAS:
    entrenamiento = db_datos[(db_datos.fecha >= '2026-03-01') & (db_datos.fecha < inicio)]
    validacion = db_datos[(db_datos.fecha >= inicio) & (db_datos.fecha < fin)]
    assert len(entrenamiento) and len(validacion)
    assert entrenamiento.y.nunique() == validacion.y.nunique() == 2
    assert entrenamiento.fecha.max() < validacion.fecha.min()
    assert not set(entrenamiento.index) & set(validacion.index)
    db_rondas.append((nombre, entrenamiento.index, validacion.index))
    for uso, grupo in [('entrenamiento', entrenamiento), ('validacion', validacion)]:
        db_particiones.append({'ronda': nombre, 'uso': uso, 'filas': len(grupo),
            'cierres': int(grupo.y.sum()), 'tasa_cierre': float(grupo.y.mean()),
            'desde': grupo.fecha.min().strftime('%Y-%m-%d'), 'hasta': grupo.fecha.max().strftime('%Y-%m-%d')})
    pesos = compute_class_weight('balanced', classes=np.array([0, 1]), y=entrenamiento.y.to_numpy())
    db_pesos.append({'ajuste': nombre, 'filas': len(entrenamiento),
        'perdidos': int((entrenamiento.y == 0).sum()), 'cierres': int(entrenamiento.y.sum()),
        'peso_perdido': float(pesos[0]), 'peso_cerrado': float(pesos[1]),
        'relacion_cerrado_perdido': float(pesos[1] / pesos[0])})

db_desarrollo = db_datos[(db_datos.fecha >= '2026-03-01') & (db_datos.fecha < '2026-07-01')]
db_julio = db_datos[(db_datos.fecha >= '2026-07-01') & (db_datos.fecha < '2026-08-01')]
assert len(db_desarrollo) + len(db_julio) == len(db_datos), 'Revisar protocolo: la fuente tiene periodos nuevos.'
assert db_desarrollo.fecha.max() < db_julio.fecha.min()
assert db_desarrollo.y.nunique() == db_julio.y.nunique() == 2
pesos_finales = compute_class_weight('balanced', classes=np.array([0, 1]), y=db_desarrollo.y.to_numpy())
db_pesos.append({'ajuste': 'final_marzo_junio', 'filas': len(db_desarrollo),
    'perdidos': int((db_desarrollo.y == 0).sum()), 'cierres': int(db_desarrollo.y.sum()),
    'peso_perdido': float(pesos_finales[0]), 'peso_cerrado': float(pesos_finales[1]),
    'relacion_cerrado_perdido': float(pesos_finales[1] / pesos_finales[0])})
for uso, grupo in [('ajuste_final', db_desarrollo), ('comprobacion_julio', db_julio)]:
    db_particiones.append({'ronda': 'final', 'uso': uso, 'filas': len(grupo),
        'cierres': int(grupo.y.sum()), 'tasa_cierre': float(grupo.y.mean()),
        'desde': grupo.fecha.min().strftime('%Y-%m-%d'), 'hasta': grupo.fecha.max().strftime('%Y-%m-%d')})
db_tabla_particiones = pd.DataFrame(db_particiones)
db_tabla_pesos = pd.DataFrame(db_pesos)
display(db_tabla_particiones)
display(db_tabla_pesos.round(3))

### 14.2. Métricas y referencias sin favorecer a una variante

La métrica de selección sigue siendo **AP**. Acompañamos con precisión, recuperación y lift en el primer 20%, además de ROC-AUC. En los empates del corte usamos la selección aleatoria esperada, igual que en el primer experimento.

El Brier se muestra como diagnóstico de las salidas, sin suponer calibración. También mostramos falsos positivos y verdaderos positivos con un corte ilustrativo de 0.5 para ver qué cambia al ponderar; **ese corte no se usa para elegir ni para ordenar la cola**. Un aumento del recall a 0.5 puede ser solo un desplazamiento de escala, sin mejorar el ranking.

FIFO desempata por ID cuando falta hora. Reglas conserva exactamente los puntos del experimento anterior. La referencia constante usa la tasa de cierre del entrenamiento correspondiente, nunca la tasa del grupo evaluado. Las comparaciones por empresa respetan la separación de colas.

In [ ]:
def db_cola(y, score):
    y, score = np.asarray(y), np.asarray(score, dtype=float)
    k = max(1, math.ceil(len(y) * DB_FRACCION))
    corte = np.sort(score)[-k]
    arriba, empate = score > corte, score == corte
    cierres = float(y[arriba].sum() + (k - arriba.sum()) * y[empate].mean())
    precision = cierres / k
    return {'k': k, 'cierres_esperados_top': cierres, 'precision_top20': precision,
        'recall_top20': cierres / y.sum() if y.sum() else np.nan,
        'lift_top20': precision / y.mean() if y.mean() else np.nan}

def db_evaluar(y, score, salida_modelo=False):
    y, score = np.asarray(y), np.asarray(score, dtype=float)
    r = {'AP': average_precision_score(y, score) if y.sum() else np.nan,
         'ROC_AUC': roc_auc_score(y, score) if len(np.unique(y)) == 2 else np.nan, **db_cola(y, score)}
    r.update({'Brier': np.nan, 'precision_corte_05': np.nan, 'recall_corte_05': np.nan,
              'falsos_positivos_corte_05': np.nan, 'verdaderos_positivos_corte_05': np.nan})
    if salida_modelo:
        pred = (score >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        r.update({'Brier': brier_score_loss(y, score),
            'precision_corte_05': precision_score(y, pred, zero_division=0),
            'recall_corte_05': recall_score(y, pred, zero_division=0),
            'falsos_positivos_corte_05': int(fp), 'verdaderos_positivos_corte_05': int(tp)})
    return r

def db_reglas(x):
    return (10 * (~x.modelo_sku.isin(['DESCONOCIDO', 'AMBIGUO', ''])).astype(int)
        + 30 * x.manifesto_cuota_inicial.eq('SI').astype(int)
        + 10 * x.forma_pago_declarada.isin(['credito', 'contado']).astype(int)
        + 40 * x.cita_observada.eq('SI').astype(int)).to_numpy() / 90

def db_fifo(grupo):
    orden = grupo.sort_values(['fecha', 'lead_id']).index
    return pd.Series(np.arange(len(grupo), 0, -1), index=orden).reindex(grupo.index).to_numpy()

def db_modelo(config):
    pesos = None if config['regimen'] == 'sin_pesos' else 'balanced'
    estimador = (LogisticRegression(C=config['parametro'], class_weight=pesos, max_iter=2000, random_state=DB_SEED)
        if config['tipo'] == 'logistica' else DecisionTreeClassifier(
            max_depth=int(config['parametro']), min_samples_leaf=40, class_weight=pesos, random_state=DB_SEED))
    return Pipeline([('categorias', OneHotEncoder(handle_unknown='ignore', sparse_output=False)), ('modelo', estimador)])

db_configs = {}
for perfil, columnas in DB_PERFILES.items():
    for tipo, parametros in [('logistica', [0.1, 1.0, 10.0]), ('arbol', [2, 3])]:
        for parametro in parametros:
            for regimen in ['sin_pesos', 'balanced']:
                candidato = f'{perfil}_{tipo}_{parametro}_{regimen}'
                db_configs[candidato] = {'candidato': candidato, 'perfil': perfil, 'columnas': columnas,
                    'tipo': tipo, 'parametro': parametro, 'regimen': regimen, 'orden': len(db_configs)}
print(f'{len(db_configs)} candidatos × {len(db_rondas)} rondas; más dos ajustes finales.')

### 14.3. Entrenar y elegir usando únicamente mayo y junio

Cada ajuste crea un pipeline nuevo. La codificación y los pesos aprenden solo de las filas de entrenamiento de esa ronda. Guardamos predicciones de validación con ID, empresa, ronda y etiqueta para poder revisar errores concretos.

La tabla de diferencias muestra `balanced − sin_pesos`: positivo favorece el balanceo para AP, precisión y lift. Un promedio favorable acompañado de un empeoramiento en un mes merece cautela. Dos rondas no bastan para afirmar estabilidad estadística.

In [ ]:
db_resultados, db_predicciones_validacion, db_referencias = [], [], []
for ronda, ix_train, ix_val in db_rondas:
    grupo = db_datos.loc[ix_val]
    for candidato, config in db_configs.items():
        modelo = db_modelo(config)
        modelo.fit(db_X.loc[ix_train, config['columnas']], db_datos.loc[ix_train, 'y'])
        score = modelo.predict_proba(db_X.loc[ix_val, config['columnas']])[:, 1]
        db_resultados.append({k: config[k] for k in ['candidato', 'perfil', 'tipo', 'parametro', 'regimen', 'orden']} |
                             {'ronda': ronda, **db_evaluar(grupo.y, score, True)})
        db_predicciones_validacion.append(grupo[['empresa_id', 'lead_id', 'fecha_registro', 'y']].assign(
            ronda=ronda, candidato=candidato, regimen=config['regimen'], score=score))
    for nombre, score, es_modelo in [
        ('reglas', db_reglas(db_X.loc[ix_val]), False),
        ('FIFO', db_fifo(grupo), False),
        ('constante', np.full(len(grupo), db_datos.loc[ix_train, 'y'].mean()), True),
    ]:
        db_referencias.append({'ronda': ronda, 'metodo': nombre, **db_evaluar(grupo.y, score, es_modelo)})

db_metricas_ronda = pd.DataFrame(db_resultados)
db_resumen = db_metricas_ronda.groupby(
    ['candidato', 'perfil', 'tipo', 'parametro', 'regimen', 'orden'], as_index=False).agg(
        AP_media=('AP', 'mean'), AP_minima=('AP', 'min'), precision_top20_media=('precision_top20', 'mean'),
        recall_top20_media=('recall_top20', 'mean'), lift_top20_medio=('lift_top20', 'mean'), Brier_medio=('Brier', 'mean'))
db_resumen = db_resumen.sort_values(['AP_media', 'orden'], ascending=[False, True], kind='stable')
db_seleccion = db_resumen.drop_duplicates('regimen', keep='first').copy()
assert set(db_seleccion.regimen) == {'sin_pesos', 'balanced'}
db_claves = ['perfil', 'tipo', 'parametro', 'ronda']
db_pareadas = db_metricas_ronda[db_metricas_ronda.regimen.eq('balanced')].merge(
    db_metricas_ronda[db_metricas_ronda.regimen.eq('sin_pesos')], on=db_claves,
    suffixes=('_balanced', '_sin_pesos'), validate='one_to_one')
for metrica in ['AP', 'precision_top20', 'recall_top20', 'lift_top20']:
    db_pareadas[f'delta_{metrica}'] = db_pareadas[f'{metrica}_balanced'] - db_pareadas[f'{metrica}_sin_pesos']
display(db_resumen.round(4))
display(db_pareadas[db_claves + [c for c in db_pareadas if c.startswith('delta_')]].round(4))
display(pd.DataFrame(db_referencias).round(4))
display(Markdown('**Representantes seleccionados sin usar julio:**'))
display(db_seleccion[['regimen', 'candidato', 'AP_media', 'precision_top20_media', 'lift_top20_medio']].round(4))

### 14.4. Reajuste y comprobación adicional en julio

Reajustamos los dos representantes con marzo–junio y mostramos julio por transparencia. No ajustamos umbrales ni cambiamos candidatos después de verlo. Si los representantes usan distinto perfil o complejidad, su diferencia combina selección de modelo y pesos; la tabla pareada anterior es la que aísla el efecto de ponderar.

La comparación global y la comparación dentro de cada empresa responden a capacidades distintas. El 20% por empresa puede sumar más filas por el redondeo. No interpretamos un buen promedio como garantía de mejora para las tres empresas.

In [ ]:
db_modelos_finales, db_scores_julio, db_evaluacion_julio, db_por_empresa = {}, {}, [], []
for fila in db_seleccion.itertuples(index=False):
    config = db_configs[fila.candidato]
    modelo = db_modelo(config)
    modelo.fit(db_X.loc[db_desarrollo.index, config['columnas']], db_desarrollo.y)
    score = modelo.predict_proba(db_X.loc[db_julio.index, config['columnas']])[:, 1]
    db_modelos_finales[fila.regimen] = {'pipeline': modelo, 'configuracion': config}
    db_scores_julio[fila.regimen] = score
    db_evaluacion_julio.append({'metodo': fila.regimen, 'candidato': fila.candidato, **db_evaluar(db_julio.y, score, True)})
for nombre, score, es_modelo in [
    ('reglas', db_reglas(db_X.loc[db_julio.index]), False),
    ('FIFO', db_fifo(db_julio), False),
    ('constante', np.full(len(db_julio), db_desarrollo.y.mean()), True),
]:
    db_scores_julio[nombre] = score
    db_evaluacion_julio.append({'metodo': nombre, 'candidato': None, **db_evaluar(db_julio.y, score, es_modelo)})

db_pred_julio = db_julio[['empresa_id', 'lead_id', 'fecha_registro', 'y']].copy()
for nombre, scores in db_scores_julio.items(): db_pred_julio[nombre] = scores
for empresa, grupo in db_pred_julio.groupby('empresa_id'):
    for nombre in db_scores_julio:
        scores = db_fifo(db_julio.loc[grupo.index]) if nombre == 'FIFO' else grupo[nombre].to_numpy()
        db_por_empresa.append({'empresa_id': empresa, 'metodo': nombre, 'filas': len(grupo),
            'cierres': int(grupo.y.sum()), **db_evaluar(grupo.y, scores, nombre in ['sin_pesos', 'balanced', 'constante'])})
db_tabla_julio = pd.DataFrame(db_evaluacion_julio)
db_tabla_empresa = pd.DataFrame(db_por_empresa)
display(db_tabla_julio.round(4))
display(db_tabla_empresa[['empresa_id', 'metodo', 'filas', 'cierres', 'AP', 'precision_top20', 'lift_top20']].round(4))

### 14.5. Visualizar la comparación y conservar la incertidumbre

El gráfico izquierdo muestra las diferencias pareadas de AP por mes: ambos modelos tienen las mismas variables y parámetros, y solo cambian los pesos. Cero significa empate. El gráfico derecho muestra el rendimiento de los dos representantes y las referencias en julio, **como comprobación adicional ya explorada**.

No convertimos una barra ligeramente más alta en una conclusión. Para decidir revisaremos consistencia entre meses, resultados por empresa, falsos positivos y utilidad de la cola. Si no mejora, esa es una conclusión válida del experimento. No cambiaremos los puntos de las reglas ni las prioridades existentes como consecuencia automática de esta ejecución.

In [ ]:
db_plot_pares = db_pareadas.assign(configuracion=lambda d:
    d.perfil + ' / ' + d.tipo + ' / ' + d.parametro.astype(str)).pivot(
        index='configuracion', columns='ronda', values='delta_AP')
fig_db, axes_db = plt.subplots(1, 2, figsize=(14, 6))
db_plot_pares.plot.barh(ax=axes_db[0], color=['#325ca8', '#249181'])
axes_db[0].axvline(0, color='black', linewidth=1)
axes_db[0].set(title='Validación: efecto de los pesos con igual configuración', xlabel='AP balanced menos AP sin pesos', ylabel='')
axes_db[0].legend(title='Mes evaluado')
db_tabla_julio.set_index('metodo').precision_top20.plot.bar(
    ax=axes_db[1], color=['#325ca8', '#249181', '#9c7937', '#8c73a5', '#999999'], rot=25)
axes_db[1].axhline(db_julio.y.mean(), color='gray', linestyle='--', label='Tasa de cierre de julio')
axes_db[1].set(title='Julio: comprobación adicional, no prueba nueva', ylabel='Precisión en el primer 20%', xlabel='')
axes_db[1].legend()
fig_db.tight_layout()
plt.show()

### 14.6. Guardar resultados para revisarlos juntos

Esta celda guarda las tablas, las predicciones, los pesos calculados, la configuración seleccionada y los dos pipelines experimentales. Todo queda en una carpeta propia de la segunda iteración. El manifiesto registra versiones, fuente, protocolo y huellas. Los archivos de esta carpeta se regeneran al repetir el bloque con las mismas fuentes.

**Estado de interpretación:** pendiente de revisión conjunta tras tu ejecución. El código selecciona representantes siguiendo el protocolo, pero no declara que el balanceo sea mejor ni promueve ningún modelo.

Para revisar: `comparacion_pareada.csv`, `resumen_validacion.csv`, `metricas_julio_explorado.csv` y `metricas_julio_empresa.csv`. Las predicciones permiten inspeccionar casos concretos. La escala de los modelos balanceados sigue sin estar calibrada como probabilidad de compra.

In [ ]:
# Controles y exportación: se ejecutarán únicamente cuando corras esta celda.
assert len(db_metricas_ronda) == len(db_configs) * len(db_rondas)
assert len(db_pareadas) == len(db_configs) // 2 * len(db_rondas)
assert all(np.isfinite(db_scores_julio[k]).all() and ((db_scores_julio[k] >= 0) & (db_scores_julio[k] <= 1)).all()
           for k in ['sin_pesos', 'balanced'])
assert hashlib.sha256(DB_FUENTE.read_bytes()).hexdigest() == db_hash_fuente
assert not set(db_desarrollo.index) & set(db_julio.index)

DB_OUT.mkdir(parents=True, exist_ok=True)
db_tablas_exportar = {
    'particiones.csv': db_tabla_particiones,
    'pesos_por_ajuste.csv': db_tabla_pesos,
    'historicos_fuera_entrenamiento.csv': db_fuera,
    'metricas_validacion_ronda.csv': db_metricas_ronda,
    'referencias_validacion.csv': pd.DataFrame(db_referencias),
    'resumen_validacion.csv': db_resumen,
    'comparacion_pareada.csv': db_pareadas,
    'seleccion_antes_julio.csv': db_seleccion,
    'predicciones_validacion.csv': pd.concat(db_predicciones_validacion, ignore_index=True),
    'metricas_julio_explorado.csv': db_tabla_julio,
    'metricas_julio_empresa.csv': db_tabla_empresa,
    'predicciones_julio_explorado.csv': db_pred_julio,
}
for nombre, tabla in db_tablas_exportar.items(): tabla.to_csv(DB_OUT / nombre, index=False, encoding='utf-8')
fig_db.savefig(DB_OUT / 'comparacion_desbalance.png', dpi=160, bbox_inches='tight')
for regimen, artefacto in db_modelos_finales.items():
    joblib.dump({**artefacto, 'version': DB_VERSION, 'uso': 'experimental_revision_pendiente'}, DB_OUT / f'modelo_{regimen}.joblib')

def db_registros(tabla): return json.loads(tabla.to_json(orient='records', force_ascii=False))
db_archivos = list(db_tablas_exportar) + ['comparacion_desbalance.png', 'modelo_sin_pesos.joblib', 'modelo_balanced.joblib']
db_manifest = {
    'version': DB_VERSION, 'semilla': DB_SEED, 'estado': 'ejecutado_pendiente_revision_conjunta',
    'fuente': str(DB_FUENTE.relative_to(DB_ROOT)), 'fuente_sha256': db_hash_fuente,
    'python': platform.python_version(),
    'paquetes': {p: metadata.version(p) for p in ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'joblib']},
    'seleccion': 'AP media de validaciones en mayo y junio; mismo presupuesto de búsqueda por régimen.',
    'julio': 'Periodo ya explorado: comprobación adicional; no se usa para seleccionar ni es una prueba nueva.',
    'particiones': db_registros(db_tabla_particiones), 'pesos': db_registros(db_tabla_pesos),
    'representantes': db_registros(db_seleccion), 'metricas_julio': db_registros(db_tabla_julio),
    'politica': 'Sin promoción automática de modelos; prioridades anteriores intactas.',
    'limitaciones': ['Datos sintéticos.', 'Periodos ya explorados; falta validación prospectiva.',
        'Sin fecha de captura de señales ni de desenlace.', 'Población entrenada: leads gestionados.',
        'Balancear no calibra probabilidades y no garantiza mejor ranking.'],
    'salidas_sha256': {p: hashlib.sha256((DB_OUT / p).read_bytes()).hexdigest() for p in db_archivos},
}
(DB_OUT / 'manifest_desbalance.json').write_text(json.dumps(db_manifest, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
display(Markdown('**Comparación ejecutada y exportada. Interpretación pendiente de revisión conjunta.** '
    'Guarda el notebook con sus salidas y avisa para revisar las tablas.'))
print('Resultados:', DB_OUT)

### 14.7. Preguntas para nuestra revisión posterior

1. Con la misma configuración, ¿balanced mejora AP y los primeros puestos tanto en mayo como en junio, o solo en un mes?
2. ¿Los representantes seleccionados usan el mismo perfil y algoritmo? Si difieren, no atribuir toda la diferencia a los pesos.
3. ¿Aumentaron los compradores recuperados al mismo costo de llamadas, o solo el recall con corte 0.5 por un cambio de escala?
4. ¿Qué ocurre dentro de cada empresa? Revisar los conteos de cierres, no solo porcentajes con muestras pequeñas.
5. ¿Julio acompaña o contradice lo observado? Recordar que ya fue explorado y que no permite una afirmación confirmatoria independiente.
6. ¿Qué decisión tomamos: mantener sin pesos, continuar investigando balanced o concluir que falta señal? Resolver después de observar los resultados, sin fabricar mejora.

**Conclusión de esta segunda iteración: documentada en 14.8 después de revisar las salidas.**

### 14.8. Revisión de la ejecución: resultado y decisión

Esta revisión utiliza exclusivamente los archivos que generó el usuario. Se verificaron las huellas de las entradas y salidas del manifiesto y las siete celdas nuevas tienen ejecución sin errores. **No se volvió a entrenar ni a ejecutar la sección.**

#### Selección y comparación

Los dos representantes son regresiones logísticas con perfil comercial y `C=1`. Por tanto, aquí sí podemos comparar directamente el cambio de pesos: coinciden las variables y los hiperparámetros.

| Indicador | Sin pesos | Balanced | Lectura |
|---|---:|---:|---|
| AP media, mayo–junio | 0.1375 | 0.1353 | Ligera ventaja sin pesos; no es prueba de significancia. |
| Compradores entre primeros 83, mayo | 8 | 8 | Mismo número de compradores; no necesariamente las mismas personas. |
| Compradores entre primeros 81, junio | 12 | 12 | Mismo rendimiento a esa capacidad. |
| AP de julio, periodo ya explorado | 0.1231 | 0.1141 | Balanced no mejora esta comprobación. |
| Compradores entre primeros 72, julio | 8 | 7 | Un comprador menos con balanced. |
| Precisión entre primeros 72, julio | 11.11% | 9.72% | Comparación al mismo costo de llamadas. |

En mayo, el AP del representante balanceado sube ligeramente (0.1230 → 0.1238); en junio baja (0.1520 → 0.1468). No hay una mejora consistente. En el ajuste final, cada comprador recibió aproximadamente **9.15 veces** el peso de un perdido: la compensación sí se aplicó.

#### Aumentar sensibilidad no equivale a mejorar el ranking

Con el corte ilustrativo de 0.5, el balanceado marca 161 leads como positivos: **16 compradores y 145 perdidos**. Recupera 16 de los 33 compradores (48.48%), pero su precisión es solo 9.94%. La tasa base de julio es aproximadamente 9.24%.

El modelo sin pesos no supera 0.5 para ningún lead. Eso no significa que no pueda ordenar: al seleccionar sus primeros 72 recupera 8 compradores. Un corte fijo de 0.5 no representa nuestra capacidad de atención y no es adecuado para decidir cuál ranking sirve más. Tampoco hay fundamento para afirmar que es imposible mejorar su corte; ese sería otro experimento, elegido sin ajustar a julio.

El Brier empeora de 0.0840 a 0.2357 al balancear. Esto muestra que las salidas crudas ponderadas se ajustan peor a la frecuencia observada como probabilidades; **no demuestra por sí solo peor ranking ni valida la calibración del modelo sin pesos**. No presentamos estas salidas como porcentajes de compra.

#### Matices: árboles y empresas

- En el perfil de ingreso, los pesos no cambiaron las métricas de ranking: solo usamos canal, que ofrece pocos niveles de ordenamiento.
- Algunos árboles comerciales balanceados sí mejoraron ligeramente la precisión del primer 20%. El árbol de profundidad 3 mejoró esa métrica en ambos meses, aunque empeoró AP en ambos. Es un intercambio real entre métricas; no corresponde decir que balancear nunca ayuda.
- El criterio fijado antes de ejecutar era AP media. No lo cambiamos después de ver las tablas para forzar otro ganador. Si la empresa fija una capacidad concreta de atención, podremos diseñar un experimento futuro que optimice esa capacidad desde el inicio.
- En julio, dentro de cada empresa, ambos representantes recuperan los mismos compradores en cantidad: **EMP-01: 2 de 26; EMP-02: 1 de 23; EMP-03: 4 de 24**. No hay mejora de captura con pesos a esa capacidad. Las primeras dos empresas tienen lift inferior a 1; el desempeño global no basta para justificar despliegue.

#### Decisión de esta iteración

**Conservar la regresión sin pesos como referencia experimental y no sustituirla por balanced con estos resultados.** Se conservan ambos artefactos y todas las tablas como evidencia del experimento. Las prioridades operativas de reglas permanecen como política provisional explicable; tampoco afirmamos que estén demostradas como superiores a FIFO.

El experimento **no muestra que el desbalance sea irrelevante**. Muestra que ponderar automáticamente las clases no produjo una mejora consistente según el criterio elegido en este conjunto. No podemos atribuir toda la debilidad del modelo al desbalance, ni demostrar solo con esto que no exista señal útil.

Julio ya se había examinado, hay pocos compradores y faltan fechas de captura y desenlace. La evidencia es exploratoria y no demuestra significancia estadística de estas diferencias. La próxima mejora debe priorizar etiquetas con horizonte definido, variables disponibles a una hora de corte y nuevos cierres para evaluar. No se activa automáticamente un nuevo modelo ni se reescriben los scores de los leads.

**Trazabilidad:** `outputs/clasificacion_04/desbalance_v2/revision_resultados.json` vincula esta lectura al manifiesto y a sus salidas. El estado «pendiente de revisión» en la salida guardada de la celda de exportación describe el momento de ejecución; esta sección documenta la revisión posterior.
